# Q3 Appendix: All-metric raw means (Accuracy / Precision / Recall / Macro-F1)

This appendix reproduces the article-ready Q3 network-transferability tables (`tab_q3_1_overall`, `tab_q3_1_per_model_transfer_drop`, and `q3_representation_overall_transfer`), but **expanded to report all four evaluation metrics** as raw mean-over-runs values, instead of only macro-F1.

The tables are presented in **tidy (long) form**: the four metrics appear as single columns — Accuracy, Precision, Recall, Macro-F1 (mean over runs) — in that order, and the transfer **regime** becomes one label column, `Regime`, with the fixed order In-network then Cross-network. Each grouping key (task, model + category, representation) therefore contributes two rows: one In-network and one Cross-network. No between-regime difference column is reported (the previous Δ MacroF1 transfer-drop column is dropped).

**Q3 invariants preserved:** drop split "Random with same distribution"; `EXCLUDED_PAIRS = {(SetA, SetC), (SetC, SetA)}` dropped from the cross-network view (same physical network, different capture period); `eval_type = In-network where train_set == test_set else Cross-network`; all 11 models; average over `run_no`. The setup melts **all four metrics** into a `metric` column and carries them through — each metric is aggregated with its own explicit metric filter. Macro-F1 values are identical to the source Q3 wide tables.


## Setup, load, melt, and run-averaging


In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', palette='muted', font_scale=1.15)

# ── Paths (notebook runs from A/) ────────────────────────────────────────────
DATA_PATH = Path('../data/wandb_export_final_hyperparameters.csv')
TAB_DIR   = Path('tables'); TAB_DIR.mkdir(exist_ok=True)

NETWORKS = ['SetA', 'SetB', 'SetC', 'SetD']
# All four metrics carried through (NOT filtered to f1_score)
METRICS  = ['f1_score', 'accuracy', 'precision', 'recall']
# Metric column display order: Accuracy, Precision, Recall, Macro-F1
METRIC_ORDER = ['accuracy', 'precision', 'recall', 'f1_score']
METRIC_LABEL = {'accuracy': 'Accuracy', 'precision': 'Precision',
                'recall': 'Recall', 'f1_score': 'Macro-F1'}

# All 11 models; category used for the per-model breakdown
DL_MODELS   = {'gru', 'lstm', 'rnn', 'mlp', 'nn'}
ALL_MODELS  = ['gru', 'knn', 'lightgbm', 'logistic-regression', 'lstm',
               'mlp', 'nn', 'random-forest', 'rnn', 'svm', 'xgboost']

# ── Excluded transfers ───────────────────────────────────────────────────────
# SetA and SetC are the SAME physical network captured at different time periods.
# The transfers SetA->SetC and SetC->SetA are excluded from every cross-network view.
EXCLUDED_PAIRS = {('SetA', 'SetC'), ('SetC', 'SetA')}

def drop_excluded_pairs(df, train_col='train_set', test_col='test_set'):
    """Return `df` without the same-network/different-time transfers (SetA<->SetC)."""
    keep = ~df.apply(lambda r: (r[train_col], r[test_col]) in EXCLUDED_PAIRS, axis=1)
    return df[keep].copy()

# ── Load raw data ────────────────────────────────────────────────────────────
raw = pd.read_csv(DATA_PATH)
raw = raw[raw['split'] != 'Random with same distribution'].copy()

# ── Parse metric columns into long form (ALL metrics) ────────────────────────
metric_cols = [c for c in raw.columns if c.count('/') == 2]
id_vars = ['model', 'task', 'split', 'enable_sequences', 'run_no']
long = raw[id_vars + metric_cols].melt(
    id_vars=id_vars, value_vars=metric_cols,
    var_name='metric_key', value_name='value')
long[['train_set', 'test_set', 'metric']] = long['metric_key'].str.split('/', expand=True)
long = long.drop(columns='metric_key')
long = long[long['train_set'].isin(NETWORKS) & long['test_set'].isin(NETWORKS)].copy()

# Flag in-network vs cross-network evaluation
long['eval_type'] = np.where(
    long['train_set'] == long['test_set'], 'In-network', 'Cross-network')

# ── Average across runs (all 4 metrics kept in the `metric` dimension) ───────
group_keys = ['model', 'task', 'split', 'enable_sequences',
              'train_set', 'test_set', 'eval_type', 'metric']
avg = (long.groupby(group_keys, as_index=False)['value']
           .mean().rename(columns={'value': 'mean_value'}))

# Exclude same-network/different-time transfers (SetA<->SetC) from ALL views
avg = drop_excluded_pairs(avg)

print(f'Long-form rows (all):    {len(long):,}')
print(f'Averaged rows (all):     {len(avg):,}')
print('Metrics present:', sorted(avg["metric"].unique()))
print('eval_type counts:')
print(avg.groupby(["eval_type", "metric"]).size())
avg.head()


Long-form rows (all):    38,528
Averaged rows (all):     3,808
Metrics present: ['accuracy', 'f1_score', 'precision', 'recall']
eval_type counts:
eval_type      metric   
Cross-network  accuracy     680
               f1_score     680
               precision    680
               recall       680
In-network     accuracy     272
               f1_score     272
               precision    272
               recall       272
dtype: int64


,model,task,split,enable_sequences,train_set,test_set,eval_type,metric,mean_value
0,gru,Binary,Random split,True,SetA,SetA,In-network,accuracy,0.941879
1,gru,Binary,Random split,True,SetA,SetA,In-network,f1_score,0.801669
2,gru,Binary,Random split,True,SetA,SetA,In-network,precision,0.801336
3,gru,Binary,Random split,True,SetA,SetA,In-network,recall,0.807207
4,gru,Binary,Random split,True,SetA,SetB,Cross-network,accuracy,0.935300


The helper below pivots the run-averaged means so that **only** the four metrics become columns (Accuracy / Precision / Recall / Macro-F1), while the transfer regime (`eval_type`) is kept as a row value in a `Regime` column (fixed order In-network then Cross-network). Each metric is computed independently over its own rows (explicit metric filter via the `metric` pivot dimension), so no metric leaks into another. No between-regime difference is produced.


In [2]:
REGIME_ORDER = ['In-network', 'Cross-network']

def expand_all_metrics(df, index_cols):
    """Tidy/long expansion: pivot ONLY `metric` into columns, keep regime as a row.

    Produces one row per (index_cols x eval_type) with a `Regime` label column and
    the four metric columns Accuracy, Precision, Recall, Macro-F1 (mean over runs),
    in that order. The regime rows follow the fixed order In-network then
    Cross-network. No between-regime difference column is produced.

    Every metric is aggregated on its own rows: the `metric` dimension is part of
    the pivot, so the mean for each metric uses only that metric's values.
    """
    g = (df.groupby(index_cols + ['eval_type', 'metric'], as_index=False)['mean_value']
           .mean())
    piv = g.pivot_table(index=index_cols + ['eval_type'], columns='metric',
                        values='mean_value')
    # Metric columns in the fixed display order Accuracy, Precision, Recall, Macro-F1
    piv = piv.reindex(columns=METRIC_ORDER)
    piv.columns = [METRIC_LABEL[m] for m in METRIC_ORDER]
    piv = piv.reset_index()
    # Regime label column, fixed order In-network then Cross-network
    piv = piv.rename(columns={'eval_type': 'Regime'})
    piv['Regime'] = pd.Categorical(piv['Regime'], categories=REGIME_ORDER, ordered=True)
    sort_cols = index_cols + ['Regime']
    piv = piv.sort_values(sort_cols).reset_index(drop=True)
    piv['Regime'] = piv['Regime'].astype(str)
    return piv


## q3_appendix_overall

Overall transferability by task — source `tab_q3_1_overall`, in tidy form. Columns: Task, Regime, Accuracy, Precision, Recall, Macro-F1. Each Task (Binary, Multiclass, All) contributes an In-network and a Cross-network row. Averaged over all models, feature approaches, and splits; the *All* rows aggregate both tasks.


In [3]:
def overall_rows(sub, label):
    r = expand_all_metrics(sub, [])
    r.insert(0, 'Task', label)
    return r

rows = []
for label, task_filter in [('Binary', 'Binary'),
                           ('Multiclass', 'Multiclass'),
                           ('All', None)]:
    sub = avg if task_filter is None else avg[avg['task'] == task_filter]
    rows.append(overall_rows(sub, label))
t_overall = pd.concat(rows, ignore_index=True)
metric_cols = [METRIC_LABEL[m] for m in METRIC_ORDER]
t_overall = t_overall[['Task', 'Regime'] + metric_cols]
t_overall[metric_cols] = t_overall[metric_cols].round(4)

t_overall.to_csv(TAB_DIR / 'q3_appendix_overall.csv', index=False)
caption = ('Q3 appendix: means over runs of Accuracy, Precision, Recall, and '
           'Macro-F1, by task and transfer regime (in-network vs. cross-network), '
           'averaged over all models, feature approaches, and splits. SetA'
           + chr(0x2194) + 'SetC transfers are excluded from cross-network (same '
           'network, different capture period).')
latex = t_overall.to_latex(index=False, float_format='%.4f', escape=False,
                           caption=caption, label='tab:q3_appendix_overall')
(TAB_DIR / 'q3_appendix_overall.tex').write_text(latex)
print('Saved q3_appendix_overall  shape:', t_overall.shape)
t_overall


Saved q3_appendix_overall  shape: (6, 6)


,Task,Regime,Accuracy,Precision,Recall,Macro-F1
0,Binary,In-network,0.9326,0.8648,0.8134,0.8220
1,Binary,Cross-network,0.7697,0.6987,0.6625,0.6066
2,Multiclass,In-network,0.9299,0.8174,0.7076,0.7273
3,Multiclass,Cross-network,0.7449,0.5498,0.5109,0.4580
4,All,In-network,0.9313,0.8411,0.7605,0.7747
5,All,Cross-network,0.7573,0.6243,0.5867,0.5323


## q3_appendix_per_model

Per-model transferability — source `tab_q3_1_per_model_transfer_drop`, in tidy form. Columns: model, Category, Regime, Accuracy, Precision, Recall, Macro-F1. A `Category` column (Deep Learning / Classical ML) is retained. Each model contributes an In-network and a Cross-network row; models are ordered by In-network Macro-F1 (descending), and within each model the regimes follow the fixed order In-network then Cross-network. Averaged over all networks, tasks, splits, and feature approaches.


In [4]:
t_model = expand_all_metrics(avg, ['model'])
# Category as in the source per-model transfer-drop table
t_model.insert(1, 'Category',
               t_model['model'].map(lambda m: 'Deep Learning' if m in DL_MODELS
                                    else 'Classical ML'))
metric_cols = [METRIC_LABEL[m] for m in METRIC_ORDER]

# Order models by In-network Macro-F1 (descending), then Regime fixed order
in_f1 = (t_model[t_model['Regime'] == 'In-network']
         .set_index('model')['Macro-F1'])
model_order = in_f1.sort_values(ascending=False).index.tolist()
t_model['model'] = pd.Categorical(t_model['model'], categories=model_order, ordered=True)
t_model['Regime'] = pd.Categorical(t_model['Regime'], categories=REGIME_ORDER, ordered=True)
t_model = (t_model.sort_values(['model', 'Regime'])
                  .reset_index(drop=True))
t_model['model'] = t_model['model'].astype(str)
t_model['Regime'] = t_model['Regime'].astype(str)

t_model = t_model[['model', 'Category', 'Regime'] + metric_cols]
t_model[metric_cols] = t_model[metric_cols].round(4)

t_model.to_csv(TAB_DIR / 'q3_appendix_per_model.csv', index=False)
caption = ('Q3 appendix: per-model means over runs of Accuracy, Precision, Recall, '
           'and Macro-F1, by transfer regime (in-network vs. cross-network), averaged '
           'over all networks, tasks, splits, and feature approaches. Models are '
           'ordered by in-network Macro-F1 (highest first). SetA' + chr(0x2194)
           + 'SetC transfers are excluded from cross-network (same network, different '
           'capture period).')
latex = t_model.to_latex(index=False, float_format='%.4f', escape=False,
                         caption=caption, label='tab:q3_appendix_per_model')
(TAB_DIR / 'q3_appendix_per_model.tex').write_text(latex)
print('Saved q3_appendix_per_model  shape:', t_model.shape)
t_model


Saved q3_appendix_per_model  shape: (22, 7)


,model,Category,Regime,Accuracy,Precision,Recall,Macro-F1
0,gru,Deep Learning,In-network,0.9760,0.8853,0.8202,0.8396
1,gru,Deep Learning,Cross-network,0.8586,0.6789,0.6429,0.5979
2,rnn,Deep Learning,In-network,0.9752,0.8944,0.8073,0.8327
3,rnn,Deep Learning,Cross-network,0.8508,0.6735,0.6370,0.5853
4,lstm,Deep Learning,In-network,0.9789,0.8976,0.8005,0.8291
5,lstm,Deep Learning,Cross-network,0.7574,0.5951,0.5863,0.4992
6,lightgbm,Classical ML,In-network,0.9218,0.8528,0.8026,0.8105
7,lightgbm,Classical ML,Cross-network,0.7950,0.6650,0.6295,0.5939
8,xgboost,Classical ML,In-network,0.9227,0.8633,0.7972,0.8088
9,xgboost,Classical ML,Cross-network,0.7929,0.6671,0.6235,0.5897


## q3_appendix_representation

Transferability by input representation — source `q3_representation_overall_transfer`, in tidy form. Columns: Representation, Regime, Accuracy, Precision, Recall, Macro-F1. Each representation (Component-based, Sequence-based) contributes an In-network and a Cross-network row (4 rows total). Averaged over all models, tasks, splits, networks, and runs.


In [5]:
APPROACHES = ['Component-based', 'Sequence-based']
rep = avg.copy()
rep['approach'] = rep['enable_sequences'].map(
    {True: 'Sequence-based', False: 'Component-based'})

rows = []
for ap in APPROACHES:
    sub = rep[rep['approach'] == ap]
    r = expand_all_metrics(sub, [])
    r.insert(0, 'Representation', ap)
    rows.append(r)
t_rep = pd.concat(rows, ignore_index=True)
metric_cols = [METRIC_LABEL[m] for m in METRIC_ORDER]
t_rep = t_rep[['Representation', 'Regime'] + metric_cols]
t_rep[metric_cols] = t_rep[metric_cols].round(4)

t_rep.to_csv(TAB_DIR / 'q3_appendix_representation.csv', index=False)
caption = ('Q3 appendix: means over runs of Accuracy, Precision, Recall, and '
           'Macro-F1, per input representation and transfer regime (in-network vs. '
           'cross-network), averaged over all models, tasks, splits, networks, and '
           'runs. SetA' + chr(0x2194) + 'SetC transfers are excluded from '
           'cross-network (same network, different capture period).')
latex = t_rep.to_latex(index=False, float_format='%.4f', escape=False,
                       caption=caption, label='tab:q3_appendix_representation')
(TAB_DIR / 'q3_appendix_representation.tex').write_text(latex)
print('Saved q3_appendix_representation  shape:', t_rep.shape)
t_rep


Saved q3_appendix_representation  shape: (4, 6)


,Representation,Regime,Accuracy,Precision,Recall,Macro-F1
0,Component-based,In-network,0.8555,0.7816,0.7245,0.7244
1,Component-based,Cross-network,0.6734,0.5834,0.5524,0.5151
2,Sequence-based,In-network,0.9726,0.8736,0.7801,0.8021
3,Sequence-based,Cross-network,0.8031,0.6465,0.6054,0.5417
